# Purpose
We expect to predict the price of a house

In [1]:
import os

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.linear_model import ElasticNet,Ridge
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split,KFold,cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler , LabelEncoder
from sklearn.metrics import root_mean_squared_error,mean_squared_error

from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

In [2]:
TRAIN_PATH = 'dataset/train.csv'
TEST_PATH = 'dataset/test.csv'
ID_COL = 'Id'
TARGET = 'SalePrice'

assert os.path.exists(TRAIN_PATH), f"The file path {TRAIN_PATH} doesnt exist. Make sure it is imported"
assert os.path.exists(TEST_PATH), f"The file path {TEST_PATH} doesnt exist. Make sure it is imported"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print(f"The shape of the train is:{train.shape}")
print(f"The shape of the train is:{test.shape}")

print(train.head())

The shape of the train is:(1460, 81)
The shape of the train is:(1459, 80)
   Id  MSSubClass MSZoning  LotFrontage  LotArea Street Alley LotShape  \
0   1          60       RL         65.0     8450   Pave   NaN      Reg   
1   2          20       RL         80.0     9600   Pave   NaN      Reg   
2   3          60       RL         68.0    11250   Pave   NaN      IR1   
3   4          70       RL         60.0     9550   Pave   NaN      IR1   
4   5          60       RL         84.0    14260   Pave   NaN      IR1   

  LandContour Utilities  ... PoolArea PoolQC Fence MiscFeature MiscVal MoSold  \
0         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
1         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      5   
2         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      9   
3         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
4         Lvl    AllPub  ...        0    NaN   NaN         NaN       0     1

In [5]:
FEATURES_COLS = train.columns
DROP_COLS = [ID_COL,TARGET]
USED_COLS = [c for c in FEATURES_COLS if c not in DROP_COLS]

# Feature/Colums handling

In [6]:

NUMERICAL_COLS = train[USED_COLS].select_dtypes(include='number').columns.tolist()
CATEGORICAL_COLS = train[USED_COLS].select_dtypes(exclude='number').columns.tolist()

In [7]:
print(f"NUMERICAL_COLS: {NUMERICAL_COLS}")
print(f"CATEGORICAL_COLS: {CATEGORICAL_COLS}")

NUMERICAL_COLS: ['MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces', 'GarageYrBlt', 'GarageCars', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVal', 'MoSold', 'YrSold']
CATEGORICAL_COLS: ['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual', 'Functional', 'FireplaceQu', 'GarageType', 'Garag

In [17]:
NUMERIC_TRANSFORMER = Pipeline(steps=[
    ('impute',SimpleImputer(strategy='mean')),
    ('scaler',StandardScaler())
])

CATEGORIC_TRANSFORMER = Pipeline(steps=[
    ('impute',(SimpleImputer(strategy='most_frequent')))
])

preprocess = ColumnTransformer(
    transformers =[
    ('num',NUMERIC_TRANSFORMER,NUMERICAL_COLS) ,
    ('cat',CATEGORIC_TRANSFORMER,CATEGORICAL_COLS)
])

model = RandomForestRegressor(
    bootstrap= False, 
    max_depth= 20, 
    max_features= 'sqrt', 
    min_samples_leaf= 1, 
    min_samples_split= 5, 
    n_estimators=200
)

model_xgb = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=200,
    max_samples=0.8,
    max_features =0.8,
    random_state=42
)
model_rf = RandomForestRegressor(
    max_depth= 3,
max_features=0.7,
max_samples= 0.7, 
n_estimators= 200
)
pipe = Pipeline(steps=[
    ('preprocess',preprocess),
    ('model',model_rf)
])
# Lets encode data
le = LabelEncoder()

for c in USED_COLS:
    train[c] = le.fit_transform(train[c])
    test[c] = le.fit_transform(test[c])

X = train[USED_COLS]
y = train[TARGET]

X_train ,X_test , y_train, y_test = train_test_split(X,y,test_size=0.2)

In [9]:
X.head()

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,5,3,36,327,1,2,3,3,0,4,...,0,0,3,4,4,0,1,2,8,4
1,0,3,51,498,1,2,3,3,0,2,...,0,0,3,4,4,0,4,1,8,4
2,5,3,39,702,1,2,0,3,0,4,...,0,0,3,4,4,0,8,2,8,4
3,6,3,31,489,1,2,0,3,0,0,...,0,0,3,4,4,0,1,0,8,0
4,5,3,55,925,1,2,0,3,0,2,...,0,0,3,4,4,0,11,2,8,4


In [ ]:
# Test with random forest
model_r = RandomForestRegressor()

params = {
    'n_estimators':[100,200,300,],
    'max_depth':[2,3],
    'max_samples':[0.7,0.9],
    'max_features':[0.7,0.8,0.9]
}
grid_seach_rf = GridSearchCV(model_r,params,cv=5)
grid_seach_rf.fit(X_train,y_train)
print(grid_seach_rf.best_params_)

In [21]:
model_xgb = xgb.XGBRegressor()

params_xgb = {
    'n_estimators':[100,200,300,],
    'max_depth':[10,14,18,20],
    'max_samples':[0.5,0.7,0.8,0.9],
    'max_features':[0.7,0.8,0.9]
    
}
grid_seach_rf = GridSearchCV(model_xgb,params_xgb,cv=5)
grid_seach_rf.fit(X_train,y_train)
print(grid_seach_rf.best_params_)

C:\Users\LENOVO T14s\AppData\Roaming\Python\Python313\site-packages\xgboost\training.py:183: UserWarning: [00:16:12] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "max_features", "max_samples" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\LENOVO T14s\AppData\Roaming\Python\Python313\site-packages\xgboost\training.py:183: UserWarning: [00:16:13] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "max_features", "max_samples" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\LENOVO T14s\AppData\Roaming\Python\Python313\site-packages\xgboost\training.py:183: UserWarning: [00:16:14] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "max_features", "max_samples" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\LENOVO T14s\AppData\Roaming\Python\Python313\site-packages\xgboost\training.py:183: UserWarning: [00:16:15] WARNING

{'max_depth': 14, 'max_features': 0.7, 'max_samples': 0.5, 'n_estimators': 100}


In [29]:
kfold = KFold(n_splits= 5,shuffle = True , random_state = 42)
f1 = cross_val_score(pipe,X,y,cv=kfold)

In [15]:
model_rf = RandomForestRegressor(
    max_depth= 3,
max_features=0.7,
max_samples= 0.7, 
n_estimators= 200
)
model_rf.fit(X_train,y_train)

y_predict = model_rf.predict(X_test)
rmse = root_mean_squared_error(y_predict,y_test)

In [16]:
rmse

38369.64145530933

In [18]:
pipe.fit(X,y)

X_test_real = test[USED_COLS]
y_predict = pipe.predict(X_test_real)
y_predict.shape

(1459,)

In [19]:
submission = pd.DataFrame({ID_COL:test[ID_COL].values,TARGET:y_predict})

SUBMIT_FILE_NAME = 'submission_h.csv'

submission.to_csv(SUBMIT_FILE_NAME,index=False)

display(submission.head())

,Id,SalePrice
0,1461,125660.394918
1,1462,140485.415260
2,1463,166071.245200
3,1464,172648.475899
4,1465,238382.642041
